In [27]:
import pandas as pd

df_final_2 = pd.read_csv("../../data/df_final_2.csv")

In [28]:
df_final_2

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,support_ru_region,igdb_id,type,first_release_date,genres,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch
0,590970,Hollow,« Меня не интересовал этот корабль... комплекс...,419.0,1.0,В основном отрицательные,231.0,0.0,NaN,NaN,...,1.0,29525,"Nintendo, YouTube, Discord, Facebook, Twitter",2017-11-16,"Shooter, Puzzle, Adventure, Indie",50.000000,3.0,2.053926e-05,NaN,1.506474e-07
1,709840,Bloody Glimpse,Bloody Glimpse will take you on a immersive jo...,42.0,1.0,Смешанные,227.0,0.0,NaN,NaN,...,1.0,59138,Twitch,2017-09-29,"Simulator, Indie",40.000000,1.0,7.839350e-07,NaN,NaN
2,741500,"John, The Zombie","Welcome to Johnwood, a weird and mysterious pl...",259.0,1.0,Смешанные,179.0,0.0,NaN,NaN,...,1.0,75333,"Nintendo, Official Website",2017-11-22,"Simulator, Indie",50.000000,0.0,1.553492e-06,NaN,NaN
3,713160,After Death,After Death is a platform / exploration game i...,299.0,1.0,Очень положительные,146.0,0.0,NaN,NaN,...,1.0,65786,NaN,2017-10-05,"Adventure, Indie",70.000000,3.0,1.168714e-05,NaN,NaN
4,463860,Shadows of Kurgansk,"Shadows of Kurgansk — игра, в которой вам пред...",200.0,1.0,Смешанные,197.0,0.0,NaN,NaN,...,1.0,26581,"Nintendo, Playstation",2016-12-15,"Role-playing (RPG), Simulator, Adventure, Indie",50.000000,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8715,318310,Choice of the Deathless,"Battle demons and undead attorneys, and win so...",200.0,1.0,Очень положительные,307.0,0.0,NaN,NaN,...,1.0,17744,"Steam, Official Website",2014-07-05,"Role-playing (RPG), Indie",77.500000,5.0,5.473167e-06,NaN,NaN
8716,818050,Wormster Dash,Prepare yourself for a one-of-its-kind platfor...,61.0,1.0,Положительные,37.0,0.0,NaN,NaN,...,1.0,89004,"Twitch, Official Website, Steam, Google Play",2018-03-29,"Adventure, Indie",44.680853,2.0,3.121362e-06,NaN,NaN
8717,709700,The Thing: Space X,At your disposal are a dozen of the best comba...,42.0,1.0,Смешанные,52.0,0.0,NaN,NaN,...,1.0,59143,Steam,2017-09-20,"Adventure, Indie",50.000000,1.0,3.905297e-06,NaN,NaN
8718,506150,Dragon Bros,Роботы заполонили собой все и уничтожают место...,199.0,1.0,Положительные,20.0,0.0,NaN,NaN,...,1.0,31970,"Steam, Official Website",2016-09-16,Indie,73.500000,1.0,1.553492e-06,NaN,NaN


### Текстовый столбец - game_description_snippet

В процессе скрапинга мы собирали краткое описание игры, тем самым это наш единственный текстовый столбец с информацией, которую в сыром выде использовать не получится. Попробуем как-нибудь преобразовать этот столбец, возможно, сможем выявить интересные наблюдения.

Попробуем применить токенизация на наши описания, но у нас часть описаний написана на русском, часть на английском - поэтому учтем это при указании стоп-слов. Также переведем весь в нижний регистр.

In [29]:
# источник - семинар с ИАДа
from nltk.tokenize import TweetTokenizer
from string import punctuation

# https://github.com/xiamx/node-nltk-stopwords/tree/master/data/stopwords
# из-за сложностей со скачиваем стоп-слов из nltk при работе локально - были взять уже готовые списки с найденного в интернете репозитория
with open('../../data/english.txt', 'r') as file:
    stopwords_english = file.read().split()

with open('../../data/russian.txt', 'r') as file:
    stopwords_russian = file.read().split()

tw = TweetTokenizer()

game_description_snippet_tokens = df_final_2['game_description_snippet'].str.lower().apply(tw.tokenize)
noise = stopwords_russian + stopwords_english + list(punctuation)
game_description_snippet_tokens = game_description_snippet_tokens.explode()
game_description_snippet_tokens = game_description_snippet_tokens[~game_description_snippet_tokens.isin(noise)]
game_description_snippet_tokens.value_counts().head(25)

game_description_snippet
game         2112
это          1178
—            1105
world        1002
игра          718
adventure     586
’             527
play          517
...           516
new           512
–             501
action        466
rpg           415
way           408
explore       389
мир           386
one           362
find          353
set           350
story         350
«             340
»             337
take          331
time          323
fight         320
Name: count, dtype: int64

Расширим вручную список стоп-слов (на основании того, что видно в выводе выше) + добавим туда слова, такие как игра, game, play - стандартный набор слов для описания **игры**

In [30]:
noise = stopwords_russian + stopwords_english + list(punctuation) + ['игра', 'игры', 'game', 'games', 'play', 'это', '—', '–', '’', '...', 'new', 'one', '«', '»']
game_description_snippet_tokens = game_description_snippet_tokens.explode()
game_description_snippet_tokens = game_description_snippet_tokens[~game_description_snippet_tokens.isin(noise)]
token_counts = game_description_snippet_tokens.value_counts()
token_counts.head(25)

game_description_snippet
world         1002
adventure      586
action         466
rpg            415
way            408
explore        389
мир            386
find           353
set            350
story          350
take           331
time           323
fight          320
experience     307
unique         307
players        296
мире           290
life           289
puzzle         289
space          276
которой        269
build          269
battle         267
shooter        260
2              254
Name: count, dtype: int64

Судя по самым популярным токенам/словам - большая часть игр связано с приключениями, определенным игровым миром, которые игроку, в часности, предстоит исследовать.

In [32]:

token_counts[token_counts > 100].tail(25)

game_description_snippet
вместе       108
party        108
”            108
действие     107
найти        107
power        106
которая      106
ancient      106
mode         106
sandbox      105
alien        105
open         104
island       104
collect      104
town         104
свои         103
death        102
simple       102
best         102
role         102
character    102
defense      102
music        102
ever         101
together     101
Name: count, dtype: int64

При рассмотрении менее популярных тегов можно заметить несколько слов, связанных, возможно, с совместными кооп/party играми ('together', 'party', 'вместе'). Не обязательно, что такие игры хуже, их просто может быть меньше, т.к. они сложнее в реализации и требуют большего охвата аудитории.